In [1]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from scripts.utils import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, AvgPool2D, Flatten, Dense, Dropout, RandomFlip, RandomRotation, RandomZoom, Rescaling, Input
from tensorflow.keras.callbacks import TensorBoard, ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import SGD, Adam
import keras_cv

C:\Users\faarc\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# directory where images are located
data_dir = "../data/original"
image_size = (32, 32)
batch_size = 32

file_paths = []
labels = []

# the name of each folder corresponds to the label of the images within
class_names = os.listdir(data_dir)
class_to_index = {name: i for i, name in enumerate(class_names)}

# for each folder, we'll save the path of each file with its correspondent label
for class_name in class_names:
    class_dir = f"{data_dir}/{class_name}"
    images = sorted(os.listdir(class_dir))
    for img_path in images:
        file_paths.append(class_dir + "/" + str(img_path))
        labels.append(class_to_index[class_name])

# now we have all the paths and labels together
file_paths = np.array(file_paths)

# bad_files = []

# for path in file_paths:
#     try:
#         if not check_jpeg(path):
#             bad_files.append(path)
#     except:
#         bad_files.append(path)

# print("invalid files:" +  str(len(bad_files)))

# todo: include if

# fix bad_files (already done)
# for path in bad_files:
#     img = Image.open(path)
#     new_path = path.rsplit(".", 1)[0] + ".jpg"
#     img.convert("RGB").save(new_path, "JPEG")

In [3]:
labels = np.array(labels)
dimension = 3

X_train, X_test, y_train, y_test = train_test_split(
    file_paths,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

# cv_train_set = tf.data.Dataset.from_tensor_slices((X_train, y_train))
# cv_train_set = cv_train_set.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
# cv_train_set = cv_train_set.shuffle(int(len(file_paths))).batch(batch_size).prefetch(tf.data.AUTOTUNE)

# cv_test_set = tf.data.Dataset.from_tensor_slices((X_test, y_test))
# cv_test_set = cv_test_set.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
# cv_test_set = cv_test_set.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [4]:
Counter(labels)

Counter({np.int64(4): 1998,
         np.int64(3): 1984,
         np.int64(7): 1709,
         np.int64(8): 1633,
         np.int64(2): 1533,
         np.int64(6): 1380,
         np.int64(5): 993,
         np.int64(0): 814,
         np.int64(1): 810,
         np.int64(9): 494})

In [5]:
class_weights = dict(enumerate(class_weights))

In [6]:
def build_baseline_model(n_filters = 64, n_neurons = 64, learning_rate = 1e-2, momentum = 0.0, 
                      kernel_size = 3, pool_size = 2, sparse = False, **kwargs):
    '''Builds the first CNN architecture. This model is intended to be very simple, to start understanding how well it performs over the dataset
        and to start developing more complex models. It uses basic concepts of CNNs: a convolution layer followed by a pooling layer, then a dense layer with a final
        softmax given output.
        
    Args:
        - n_filters (int): number of filters to be used on convolution layers (Conv2D). Default set to 64.
        - n_neurons (int): number of neurons to be used on dense layers (Dense). Default set to 64.
        - learning_rate (float): indicates a which rate the gradient moves. Default set to 1e-2.
        - momentum (float): proportion regarding how much of past gradients affect the current gradient step. Default set to 0.0.
        - kernel_size (int): indicates the size of the squared sliding window over the feature map. Default set to 3.
        - pool_size (int): indicated the size of the downsample pooling window. Default set to 2.
        - sparse (bool): sets if labels should be considered as one-hot (False) or integers (True). Default set to True.
        - other parameters such as activation function (ReLU), padding (same) and activation (softmax) were kept as default throught all architectures.
    Returns:
        - model: fully compiled model with all respective features.
    
    '''
    
    normalization_layer = Rescaling(1./255)
    
    model = Sequential([
    Input(shape = (image_size[0], image_size[1], dimension)),
    normalization_layer,
    Conv2D(n_filters, kernel_size, activation = "relu", padding = "same"),
    MaxPooling2D(pool_size),
    Flatten(),
    Dense(n_neurons, activation="relu"),
    Dense(10, activation = "softmax")
                    ])
    
    # for the first model only SGD optimizer is used.
    optimizer = SGD(learning_rate=learning_rate, momentum=momentum)
    
    loss = "categorical_crossentropy"
    if sparse:
        loss = "sparse_categorical_crossentropy"
    model.compile(loss = loss,
        optimizer = optimizer,
        metrics = ["accuracy"])
    
    return model

In [11]:
# X_train_array, y_train_array = pass_batchs_to_arrays(cv_train_set, pass_dummy=True)
# X_test_array, y_test_array = pass_batchs_to_arrays(cv_test_set, pass_dummy=True)

param_grid = {"learning_rate": [1e-2, 1e-3],
              "momentum": [0.0,0.9],
              "kernel_size": [2, 3, 5],
              "n_filters": [32, 64],
              "n_neurons": [32, 64]}

# X_train, X_test, y_train, y_test
best_params, best_accuracy, mean_scores  = CNN_GridSearchCV(X_train, y_train, param_grid, build_baseline_model, 42, 5, n_jobs = -1)

KeyboardInterrupt: 

In [9]:
best_params

{'learning_rate': 0.01,
 'momentum': 0.0,
 'kernel_size': 2,
 'n_filters': 32,
 'n_neurons': 32}

In [10]:
best_accuracy

np.float64(0.5594669580459595)

In [24]:
which_arch = "first_architecture"

run_id, run_logdir = get_run_logdir(which_arch, "1")

# different callbacks to: save the best model over validation set, save logs of loss and accuracy for later visualization and early stopping.
model_checkpoint = ModelCheckpoint(run_logdir +"\\first_model.keras", save_best_only=True)
tensorboard_cb = TensorBoard(run_logdir)
early_stopping = EarlyStopping(patience = 10, restore_best_weights=True)

# compile the model
model = build_first_model(**best_params, sparse= False)

# train
# we use a higher number of epochs since early stopping should stop since it reaches a no-improvement point.
model.fit(cv_train_set, validation_data=cv_val_set, epochs=100, callbacks= [tensorboard_cb,model_checkpoint, early_stopping])

table_from_history(model.history.history, run_id).to_csv("curves_data/"+which_arch+"/"+run_id+".csv")

# with the best parameters achieved, we train the first model and evaluate for the test error
save_params(run_logdir, "best_params_first_model.pkl",best_params)

Epoch 1/100
334/334 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.1706 - loss: 2.2087 - val_accuracy: 0.2479 - val_loss: 2.0480
Epoch 2/100
334/334 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.2721 - loss: 2.0007 - val_accuracy: 0.3139 - val_loss: 1.8953
Epoch 3/100
334/334 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.3544 - loss: 1.8404 - val_accuracy: 0.3970 - val_loss: 1.7372
Epoch 4/100
334/334 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.4104 - loss: 1.7221 - val_accuracy: 0.3978 - val_loss: 1.6751
Epoch 5/100
334/334 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.4377 - loss: 1.6362 - val_accuracy: 0.4427 - val_loss: 1.6000
Epoch 6/100
334/334 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.4576 - loss: 1.5621 - val_accuracy: 0.4659 - val_loss: 1.5368
Epoch 7/100
334/334 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.4929 - loss: 1.4737 - val_accuracy: 0.4742 - val_loss: 1.4819
Epoch 8/100
334/334 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.5072 - loss: 1.4132 - 

In [25]:
model.evaluate(cv_test_set)

42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5858 - loss: 1.2522


[1.2392598390579224, 0.592509388923645]

In [26]:
y_pred = np.argmax(model.predict(cv_test_set), axis = 1)

42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


In [29]:
cm = compute_classification_metrics(y_test_array, y_pred, class_names)

              precision    recall  f1-score   support

     battery       0.56      0.77      0.65        81
  biological       0.50      0.78      0.61        81
   cardboard       0.64      0.61      0.63       153
     clothes       0.72      0.80      0.76       198
       glass       0.49      0.68      0.57       200
       metal       0.50      0.17      0.26        99
       paper       0.50      0.45      0.47       138
     plastic       0.66      0.50      0.57       171
       shoes       0.72      0.58      0.64       164
       trash       0.58      0.38      0.46        50

    accuracy                           0.59      1335
   macro avg       0.59      0.57      0.56      1335
weighted avg       0.60      0.59      0.58      1335

